# 02 — RFM Feature Engineering

**Source:** `outputs/retail_clean.parquet`  
**SQL:** `queries/rfm.sql` (Queries 1 & 2)  
**Output:** `outputs/rfm_features.parquet`

RFM features per customer:

| Feature | Description |
|---|---|
| `recency` | Days since last purchase (vs snapshot date 2011-12-09) |
| `frequency` | Distinct invoice count |
| `monetary` | Total revenue (Quantity × Price, GBP) |
| `aov` | Average order value = monetary / frequency |
| `purchase_velocity` | Invoices per day of customer lifetime |
| `days_as_customer` | Days between first and last purchase |
| `unique_products` | Distinct StockCodes purchased |
| `unique_countries` | Distinct countries in transaction history |

In [1]:
import pandas as pd
import numpy as np
import duckdb
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

In [2]:
PARQUET_PATH = Path("../outputs/retail_clean.parquet")
SQL_PATH     = Path("../queries/rfm.sql")
OUTPUT_PATH  = Path("../outputs/rfm_features.parquet")

assert PARQUET_PATH.exists(), f"Run notebook 01 first — {PARQUET_PATH} not found"
assert SQL_PATH.exists(),     f"SQL file not found: {SQL_PATH}"

con = duckdb.connect()
con.execute(
    f"CREATE TABLE transactions AS SELECT * FROM read_parquet('{PARQUET_PATH.resolve().as_posix()}')"
)
result = con.execute("SELECT COUNT(*) FROM transactions").fetchone()
n = result[0] if result else 0
print(f"Loaded {n:,} rows into DuckDB as 'transactions'")

Loaded 779,425 rows into DuckDB as 'transactions'


In [3]:
# Parse queries/rfm.sql — split on ';\n', keep blocks that contain SELECT or WITH
sql_text = SQL_PATH.read_text()
queries  = [
    q.strip() for q in sql_text.split(';\n')
    if q.strip() and any(kw in q.upper() for kw in ('SELECT', 'WITH'))
]
q1_sql, q2_sql, q3_sql = queries[:3]
print(f"Parsed {len(queries)} queries from {SQL_PATH.name}")

Parsed 3 queries from rfm.sql


---
## Section 1 — RFM Base (Query 1)

Recency, Frequency, Monetary per customer — the three core features.

In [4]:
rfm_base = con.execute(q1_sql).df()
print(f"RFM base : {rfm_base.shape[0]:,} customers  ×  {rfm_base.shape[1]} features")
max_date = con.execute('SELECT MAX(InvoiceDate) FROM transactions').fetchone()
max_date = max_date[0] if max_date else "N/A"
print(f"\nSnapshot date (max InvoiceDate): {max_date}")
rfm_base.head()

RFM base : 5,878 customers  ×  4 features

Snapshot date (max InvoiceDate): 2011-12-09 12:50:00


,customer_id,recency,frequency,monetary
0,12346,325,12,77556.46
1,12347,2,8,4921.53
2,12348,75,5,2019.40
3,12349,18,4,4428.69
4,12350,310,1,334.40


---
## Section 2 — Extended Features (Query 2)

Adds AOV, purchase velocity, customer tenure, product breadth, and geography.

In [5]:
rfm = con.execute(q2_sql).df()
print(f"Extended features : {rfm.shape[0]:,} customers  ×  {rfm.shape[1]} features")
rfm.head()

Extended features : 5,878 customers  ×  9 features


,customer_id,recency,frequency,monetary,aov,purchase_velocity,days_as_customer,unique_products,unique_countries
0,12346,325,12,77556.46,6463.038333,0.029925,400,27,1
1,12347,2,8,4921.53,615.191250,0.019851,402,126,1
2,12348,75,5,2019.40,403.880000,0.013736,363,25,1
3,12349,18,4,4428.69,1107.172500,0.006993,571,138,1
4,12350,310,1,334.40,334.400000,1.000000,0,17,1


In [6]:
rfm.describe().round(2)

,customer_id,recency,frequency,monetary,aov,purchase_velocity,days_as_customer,unique_products,unique_countries
count,5878.00,5878.00,5878.00,5878.00,5878.00,5878.00,5878.00,5878.00,5878.00
mean,15315.31,200.87,6.29,2955.90,385.18,0.32,273.39,81.99,1.00
std,1715.57,209.35,13.01,14440.85,1214.29,0.48,258.96,116.48,0.05
min,12346.00,0.00,1.00,2.95,2.95,0.00,0.00,1.00,1.00
25%,13833.25,25.00,1.00,342.28,176.68,0.01,0.00,19.00,1.00
50%,15314.50,95.00,3.00,867.74,279.24,0.03,221.00,45.00,1.00
75%,16797.75,379.00,7.00,2248.30,414.90,1.00,512.00,103.00,1.00
max,18287.00,738.00,398.00,580987.04,84236.25,3.00,738.00,2550.00,2.00


---
## Section 3 — Feature Distributions

RFM distributions are typically highly right-skewed. Log scale on monetary and AOV axes
reveals the shape of the distribution more clearly than linear scale.

In [7]:
fig = px.histogram(
    rfm, x="recency", nbins=60,
    title="Recency — days since last purchase (lower = more recent)",
    labels={"recency": "Days", "count": "Customers"},
)
fig.update_layout(bargap=0.05)
fig.show()

In [8]:
fig = px.histogram(
    rfm, x="frequency", nbins=60,
    title="Frequency — distinct invoice count per customer",
    labels={"frequency": "Invoice count", "count": "Customers"},
)
fig.update_layout(bargap=0.05)
fig.show()

In [9]:
fig = px.histogram(
    rfm, x="monetary", nbins=80, log_x=True,
    title="Monetary — total revenue per customer (log scale, GBP)",
    labels={"monetary": "Revenue GBP (log)", "count": "Customers"},
)
fig.update_layout(bargap=0.05)
fig.show()

In [10]:
fig = px.histogram(
    rfm, x="aov", nbins=60, log_x=True,
    title="Average Order Value — monetary / frequency (log scale, GBP)",
    labels={"aov": "AOV GBP (log)", "count": "Customers"},
)
fig.update_layout(bargap=0.05)
fig.show()

In [11]:
# RFM scatter: recency vs monetary, sized by frequency
fig = px.scatter(
    rfm.sample(min(2000, len(rfm)), random_state=42),
    x="recency", y="monetary",
    size="frequency", color="frequency",
    log_y=True,
    title="Recency vs Monetary (size & colour = frequency, log-y, sample n=2000)",
    labels={"recency": "Recency (days)", "monetary": "Monetary GBP (log)", "frequency": "Frequency"},
    color_continuous_scale="Viridis",
    opacity=0.6,
)
fig.show()

In [12]:
rfm.to_parquet(OUTPUT_PATH, engine="pyarrow", index=False)

# Smoke-check
check = pd.read_parquet(OUTPUT_PATH)
assert len(check) == len(rfm),            "Row count mismatch"
assert (check["recency"]   >= 0).all(),   "recency must be >= 0"
assert (check["frequency"] >= 1).all(),   "frequency must be >= 1"
assert (check["monetary"]  >  0).all(),   "monetary must be > 0"
assert check["customer_id"].nunique() == len(check), "customer_id must be unique"

print(f"Saved  → {OUTPUT_PATH.resolve()}")
print(f"Size   : {OUTPUT_PATH.stat().st_size / 1024 / 1024:.2f} MB")
print(f"Shape  : {check.shape[0]:,} customers  ×  {check.shape[1]} features")
print("\nAll assertions passed.")

Saved  → C:\Users\h11la\OneDrive\Documents\00 Portfolio\customer-analytics-ml-pipeline\retail-clv-churn-prediction\outputs\rfm_features.parquet
Size   : 0.18 MB
Shape  : 5,878 customers  ×  9 features

All assertions passed.
